# 03 — Implementasi TF-IDF DapurFinder

## Tujuan
Notebook ini digunakan untuk membangun dasar mesin pencari resep pada DapurFinder menggunakan metode TF-IDF.

TF-IDF digunakan untuk memberikan bobot pada kata dalam teks resep. Bobot tersebut kemudian dapat digunakan untuk membantu sistem menemukan resep yang relevan dengan kata kunci pengguna.

Pada tahap ini, kita akan:
1. Membaca dataset resep yang telah dibersihkan.
2. Menyiapkan teks resep untuk pencarian.
3. Mengubah teks menjadi representasi numerik menggunakan TF-IDF.
4. Menguji pencarian resep berdasarkan kata kunci.

Hasil dari tahap ini nantinya akan digunakan sebagai dasar pembuatan website DapurFinder menggunakan Streamlit.

In [1]:

import pandas as pd
from pathlib import Path

# Menentukan folder utama proyek
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Membaca dataset hasil preprocessing
DATA_PATH = ROOT_DIR / "dataset" / "recipes_clean.csv"
df = pd.read_csv(DATA_PATH)

print("Lokasi dataset:", DATA_PATH)
print("Jumlah resep:", len(df))
print("Jumlah kolom:", len(df.columns))
print("\nNama kolom:")
print(df.columns.tolist())

df[["Title", "Ingredients", "Steps", "Category"]].head()


Lokasi dataset: D:\STKI-Recipe-Finder\dataset\recipes_clean.csv
Jumlah resep: 14943
Jumlah kolom: 6

Nama kolom:
['Title', 'Ingredients', 'Steps', 'Loves', 'URL', 'Category']


,Title,Ingredients,Steps,Category
0,Ayam Woku Manado,1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...,Cuci bersih ayam dan tiriskan. Lalu peras jeru...,ayam
1,Ayam goreng tulang lunak,1 kg ayam (dipotong sesuai selera jangan kecil...,"Haluskan bumbu2nya (BaPut, ketumbar, kemiri, k...",ayam
2,Ayam cabai kawin,1/4 kg ayam--3 buah cabai hijau besar--7 buah ...,Panaskan minyak di dalam wajan. Setelah minyak...,ayam
3,Ayam Geprek,250 gr daging ayam (saya pakai fillet)--Secuku...,Goreng ayam seperti ayam krispi--Ulek semua ba...,ayam
4,Minyak Ayam,400 gr kulit ayam & lemaknya--8 siung bawang p...,Cuci bersih kulit ayam. Sisihkan--Ambil 50 ml ...,ayam


## Persiapan Teks untuk Pencarian

Pada tahap ini, judul resep dan bahan-bahan digabungkan menjadi satu teks pencarian. Judul membantu sistem mengenali nama masakan, sedangkan bahan membantu sistem menemukan resep berdasarkan bahan yang dimiliki pengguna.

Kolom Steps tetap dipertahankan untuk informasi detail resep, tetapi tidak dimasukkan ke dalam teks pencarian awal.

In [2]:

# Memastikan kolom yang dibutuhkan tersedia
kolom_pencarian = ["Title", "Ingredients"]

if not all(kolom in df.columns for kolom in kolom_pencarian):
    raise ValueError("Kolom Title atau Ingredients tidak ditemukan.")

# Menggabungkan judul dan bahan menjadi satu teks pencarian
df["search_text"] = (
    df["Title"].fillna("").astype(str)
    + " "
    + df["Ingredients"].fillna("").astype(str)
)

print("Jumlah teks pencarian:", len(df["search_text"]))
print("\nContoh teks pencarian pertama:")
print(df.loc[0, "search_text"])


Jumlah teks pencarian: 14943

Contoh teks pencarian pertama:
Ayam Woku Manado 1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk Nipis--2 Sdm Garam--3 Ruas Kunyit--7 Bawang Merah--7 Bawang Putih--10 Cabe Merah--10 Cabe Rawit Merah (sesuai selera)--3 Butir Kemiri--2 Batang Sereh--2 Lembar Daun Salam--2 Ikat Daun Kemangi--Penyedap Rasa--1 1/2 Gelas Air--


## Implementasi TF-IDF

TF-IDF (*Term Frequency–Inverse Document Frequency*) adalah metode untuk memberikan bobot pada kata berdasarkan kemunculannya dalam sebuah resep dan seberapa umum kata tersebut pada seluruh kumpulan resep.

Kata yang cukup penting untuk membedakan resep dapat memperoleh bobot lebih besar, sedangkan kata yang muncul di hampir semua resep cenderung kurang membedakan.

Pada DapurFinder, TF-IDF digunakan untuk mengubah teks judul dan bahan menjadi representasi numerik agar dapat dibandingkan dengan kata kunci pengguna.

In [3]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Membuat objek TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

# Mengubah teks resep menjadi matriks TF-IDF
tfidf_matrix = tfidf_vectorizer.fit_transform(df["search_text"])

print("Jumlah resep:", tfidf_matrix.shape[0])
print("Jumlah fitur kata/frasa:", tfidf_matrix.shape[1])
print("Bentuk matriks TF-IDF:", tfidf_matrix.shape)


Jumlah resep: 14943
Jumlah fitur kata/frasa: 50000
Bentuk matriks TF-IDF: (14943, 50000)


## Implementasi Pencarian dengan Cosine Similarity

Cosine Similarity digunakan untuk mengukur kemiripan antara vektor kata kunci pengguna dan vektor setiap resep. Sistem kemudian mengurutkan resep dari nilai kemiripan tertinggi ke terendah.

Dengan metode ini, pengguna dapat mencari resep berdasarkan nama masakan maupun bahan yang tersedia.

In [4]:

from sklearn.metrics.pairwise import cosine_similarity

def cari_resep(kata_kunci, jumlah_hasil=10):
    # Memastikan kata kunci tidak kosong
    if not isinstance(kata_kunci, str) or not kata_kunci.strip():
        print("Masukkan kata kunci pencarian terlebih dahulu.")
        return pd.DataFrame()

    # Mengubah kata kunci menjadi vektor TF-IDF
    query_vector = tfidf_vectorizer.transform([kata_kunci])

    # Menghitung kemiripan dengan seluruh resep
    skor = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Mengambil indeks resep dengan skor tertinggi
    indeks_teratas = skor.argsort()[::-1]
    indeks_teratas = [
        i for i in indeks_teratas
        if skor[i] > 0
    ][:jumlah_hasil]

    # Menyusun hasil pencarian
    hasil = df.iloc[indeks_teratas][
        ["Title", "Ingredients", "Steps", "Category", "URL"]
    ].copy()

    hasil["Skor Kemiripan"] = skor[indeks_teratas]

    return hasil.reset_index(drop=True)

print("Fungsi pencarian resep berhasil dibuat.")


Fungsi pencarian resep berhasil dibuat.


In [5]:
hasil = cari_resep("ayam cabai", jumlah_hasil=10)

hasil[["Title", "Category", "Skor Kemiripan"]]

,Title,Category,Skor Kemiripan
0,Ayam cabai kawin,ayam,0.408402
1,Schotel telor,telur,0.310724
2,Semur Telor Simpel,telur,0.307795
3,Pepes Ayam Bumbu Rempah,ayam,0.275612
4,Kepala Ayam Cabai Hijau,ayam,0.275560
5,Ayam Cabai Hijau,ayam,0.268308
6,Ayam cabai hijau,ayam,0.263883
7,Ayam cabai hijau,ayam,0.198842
8,Menu diet kentang dan telur tamagoyaki,telur,0.192687
9,Ayam cabai ijo,ayam,0.185911


In [6]:

daftar_uji = [
    "ayam",
    "tempe kecap",
    "telur dadar",
    "ikan pedas",
    "sapi lada hitam"
]

for kueri in daftar_uji:
    print(f"\n{'=' * 50}")
    print("Kata kunci:", kueri)

    hasil_uji = cari_resep(kueri, jumlah_hasil=5)

    if not hasil_uji.empty:
        display(
            hasil_uji[["Title", "Category", "Skor Kemiripan"]]
        )
    else:
        print("Tidak ada resep yang cocok.")



Kata kunci: ayam


,Title,Category,Skor Kemiripan
0,Ayam goreng Cihuy,ayam,0.247760
1,Ayam Goreng minim bumbu,ayam,0.191821
2,Ayam goreng bumbu,ayam,0.190916
3,Ayam saus tiram,ayam,0.190541
4,Ayam kentuky cripy,ayam,0.186419



Kata kunci: tempe kecap


,Title,Category,Skor Kemiripan
0,Tempe manis,tempe,0.450452
1,Tempe Kecap,tempe,0.409353
2,Tempe kecap,tempe,0.344887
3,Soun tempe kecap,tempe,0.337009
4,Tempe Kecap,tempe,0.329712



Kata kunci: telur dadar


,Title,Category,Skor Kemiripan
0,Telur dadar sajiku,telur,0.453622
1,Telur dadar warteg,telur,0.433711
2,Telur Dadar Gulung,telur,0.433106
3,Telur Dadar,telur,0.429463
4,Telur dadar,telur,0.400002



Kata kunci: ikan pedas


,Title,Category,Skor Kemiripan
0,Ikan kembung asem pedas,ikan,0.207421
1,Ikan tongkol suwir pedas,ikan,0.187264
2,Ikan Kembung Tahu sambal pedas manis,ikan,0.183356
3,Oseng Pedas Ikan Kembung,ikan,0.180784
4,Ikan sarden pedas manis,ikan,0.180353



Kata kunci: sapi lada hitam


,Title,Category,Skor Kemiripan
0,Sapi lada hitam,sapi,0.461403
1,Mie Sapi Lada Hitam,sapi,0.459089
2,Daging sapi lada hitam dapur mama,sapi,0.416356
3,Sapi Lada Hitam,sapi,0.416102
4,Sapi Lada Hitam,sapi,0.412178


In [7]:

def cari_resep(kata_kunci, jumlah_hasil=10):
    # Validasi kata kunci
    if not isinstance(kata_kunci, str) or not kata_kunci.strip():
        print("Masukkan kata kunci pencarian terlebih dahulu.")
        return pd.DataFrame()

    # Validasi jumlah hasil
    if not isinstance(jumlah_hasil, int) or jumlah_hasil < 1:
        raise ValueError("jumlah_hasil harus berupa bilangan bulat positif.")

    # Batasi hasil maksimum agar pencarian tetap ringan
    jumlah_hasil = min(jumlah_hasil, len(df))

    # Mengubah kata kunci menjadi vektor TF-IDF
    query_vector = tfidf_vectorizer.transform([kata_kunci])

    # Jika kata kunci tidak memiliki kata yang dikenali model
    if query_vector.nnz == 0:
        print("Kata kunci tidak dikenali. Coba gunakan nama masakan atau bahan lain.")
        return pd.DataFrame()

    # Hitung kemiripan dengan seluruh resep
    skor = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Ambil hasil dengan skor positif tertinggi
    indeks_teratas = skor.argsort()[::-1]
    indeks_teratas = [i for i in indeks_teratas if skor[i] > 0][:jumlah_hasil]

    hasil = df.iloc[indeks_teratas][
        ["Title", "Ingredients", "Steps", "Category", "URL"]
    ].copy()

    hasil["Skor Kemiripan"] = skor[indeks_teratas]

    return hasil.reset_index(drop=True)

print("Fungsi pencarian DapurFinder berhasil diperbarui.")


Fungsi pencarian DapurFinder berhasil diperbarui.


In [8]:

# Uji kata kunci kosong
hasil_kosong = cari_resep("   ")

# Uji kata yang tidak relevan dengan isi dataset
hasil_tidak_dikenal = cari_resep("xyzqwerty123")

# Uji pencarian normal
hasil_normal = cari_resep("tempe kecap", jumlah_hasil=5)

if not hasil_normal.empty:
    display(hasil_normal[["Title", "Category", "Skor Kemiripan"]])


Masukkan kata kunci pencarian terlebih dahulu.
Kata kunci tidak dikenali. Coba gunakan nama masakan atau bahan lain.


,Title,Category,Skor Kemiripan
0,Tempe manis,tempe,0.450452
1,Tempe Kecap,tempe,0.409353
2,Tempe kecap,tempe,0.344887
3,Soun tempe kecap,tempe,0.337009
4,Tempe Kecap,tempe,0.329712
